# 用 gpt-oss 和 Hugging Face Transformers 做 fine-tuning

> 原文：[articles/gpt-oss/fine-tune-transfomers.ipynb](https://github.com/openai/openai-cookbook/blob/main/articles/gpt-oss/fine-tune-transfomers.ipynb)



## 這篇在做什麼
用 Hugging Face 的 TRL 加 LoRA，把 `openai/gpt-oss-20b` 做 supervised fine-tuning，讓它能用你指定的語言寫 chain-of-thought。訓練資料是一份 1,000 筆的多語言 reasoning 資料集。整個流程從裝套件、整理資料、載入模型、訓練一直到推論都有。

## 學完你會
- 看懂 Harmony 格式和 chat template 怎麼把 `messages` 變成模型吃的文字，包含 `analysis` 和 `final` channel。
- 用 `LoraConfig` 設定 LoRA，包含 MoE 專家層的 `target_parameters`，再用 `SFTTrainer` 訓練。
- 把訓好的 adapter 合併回 base model，做出「問題用西班牙文、reasoning 用德文」這種混合語言的推論。

## 開跑前
- 需要 Hugging Face 帳號和 token，最後要 push 到 Hub。
- 預設設計給一張 80GB 的 H100。Colab 的話要選有大 VRAM 的 GPU runtime，否則得調小 batch size 和 sequence length。

In [ ]:
# 環境設定：在 Colab 左側 🔑 Secrets 新增 OPENAI_API_KEY
%pip install -q -U datasets huggingface_hub peft torch transformers trl
import os
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
except ImportError:
    pass  # 本機執行就用自己的環境變數

# 抓原 repo 的資料檔（只下載這個資料夾）
if not os.path.exists('openai-cookbook'):
    !git clone -q --depth 1 --filter=blob:none --sparse https://github.com/openai/openai-cookbook
    !cd openai-cookbook && git sparse-checkout set articles/gpt-oss
os.chdir('openai-cookbook/articles/gpt-oss')

# 用 Hugging Face fine-tune 一個多語言 Reasoner

作者：[Edward Beeching](https://huggingface.co/edbeeching)、[Quentin Gallouédec](https://huggingface.co/qgallouedec)、[Lewis Tunstall](https://huggingface.co/lewtun)

像 [OpenAI o3](https://openai.com/index/introducing-o3-and-o4-mini/) 這類大型 reasoning 模型，會先產生 chain-of-thought 來提升回答的準確度和品質。但多數模型就算你用別的語言問，reasoning 還是用英文。

這份 notebook 示範怎麼把 OpenAI 的開放權重 reasoning 模型 [OpenAI gpt-oss-20b](https://huggingface.co/openai/gpt-oss-20b) fine-tune 成能用多種語言做 reasoning。做法是在模型的 system prompt 加一個新的 _"reasoning language"_ 選項，再用 Hugging Face 的 [TRL library](https://github.com/huggingface/trl) 在多語言 reasoning 資料集上做 [supervised fine-tuning](https://huggingface.co/learn/llm-course/chapter11/1)。

步驟如下：

1. **Setup：** 安裝需要的套件。
2. **準備資料集：** 下載資料集並整理成 fine-tuning 要的格式。
3. **準備模型：** 載入 base model，並用 [LoRA](https://huggingface.co/learn/llm-course/chapter11/4)（一種省記憶體的技巧）設定好 fine-tuning。
4. **Fine-tuning：** 用多語言 reasoning 資料訓練模型。
5. **推論：** 用 fine-tune 後的模型產生不同語言的 reasoning 回應。

最後會得到一個多語言 reasoning 模型，可以用英文、西班牙文、法文、義大利文或德文產生 chain-of-thought。語言還能混著用，例如用西班牙文提問，要求用德文 reasoning，最後回答又是西班牙文：

```txt
User:
    ¿Cuál es el capital de Australia?
Assistant reasoning:
    Okay, der Benutzer fragt nach der Hauptstadt Australiens. Ich erinnere mich, dass Canberra die Hauptstadt ist. Ich
    sollte das bestätigen. Lass mich sehen, ob es irgendwelche potenziellen Verwirrungen gibt. Der Benutzer könnte auch
    an der größten Stadt interessiert sein. Die größte Stadt ist Sydney, aber die Hauptstadt ist Canberra. Ich sollte
    das klarstellen. Vielleicht auch erwähnen, dass Canberra eine geplante Stadt ist und nicht die größte. Der Benutzer
    könnte auch nach der Geografie fragen. Vielleicht erwähne ich, dass Canberra im südwestlichen Teil der Australian
    Capital Territory liegt. Ich sollte die Antwort präzise und freundlich halten. Vielleicht auch erwähnen, dass
    Canberra oft mit Sydney verwechselt wird. Ich sollte sicherstellen, dass die Antwort klar und korrekt ist.
Assistant response:
    La capital de Australia es **Canberra**. Aunque es la ciudad más pequeña de las principales capitales del país, fue
    elegida en 1908 como la sede del gobierno federal para equilibrar la influencia entre las ciudades de Sydney y
    Melbourne. Canberra está ubicada en el Territorio de la Capital Australiana (ACT), en el este de Australia.
```

希望這份教學能幫到使用小眾語言的 AI 開發者，讓他們用母語就能看懂 [`openai/gpt-oss-20b`](https://huggingface.co/openai/gpt-oss-20b) 的思考過程。

> **Note：** 這份 notebook 設計成在單張 80GB 記憶體的 H100 上跑。如果你的 GPU 比較小，可以把下面超參數裡的 batch size 和 sequence length 調小。

## Setup

先把需要的套件裝好。第一步先裝 PyTorch（在 Colab 上通常已經預裝，可以直接跳過或確認版本就好）：

In [ ]:
%pip install torch --index-url https://download.pytorch.org/whl/cu128

接著裝其他相依套件：

In [ ]:
%pip install "trl>=0.20.0" "peft>=0.17.0" "transformers>=4.55.0" trackio

最後登入你的 Hugging Face 帳號（Colab 上可以把 token 存在 Secrets，或直接貼 token）：

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

套件裝好了，來看看 fine-tuning 要用的資料集。

## 準備資料集

我們用 [Multilingual-Thinking](https://huggingface.co/datasets/HuggingFaceH4/Multilingual-Thinking)，這是一份 reasoning 資料集，chain-of-thought 已經被翻成法文、西班牙文、德文等多種語言。用它 fine-tune `openai/gpt-oss-20b` 之後，模型就學會用這些語言產生 reasoning 步驟，講這些語言的使用者也能看懂它的思考過程。

<iframe
  src="https://huggingface.co/datasets/HuggingFaceH4/Multilingual-Thinking/embed/viewer/default/train"
  frameborder="0"
  width="100%"
  height="560px"
></iframe>



從 Hugging Face Hub 下載這份資料集：

In [ ]:
from datasets import load_dataset

dataset = load_dataset("HuggingFaceH4/Multilingual-Thinking", split="train")
dataset

這份資料集只有 1,000 筆，不過對 `openai/gpt-oss-20b` 這種已經做過大量 post-training 的模型來說，通常綽綽有餘。來看一筆訓練範例：

In [ ]:
dataset[0]

`gpt-oss` 系列是用 Harmony response format 訓練的，這個格式用來定義對話結構、產生 reasoning 輸出和組織 function call。它設計得跟 OpenAI Responses API 很像，下表整理了資料集中用到的幾種訊息類型：

|||
| :---- | :--|
| `developer` | The developer message is used to provide custom instructions for the model (what we usually call the `system` role). |
| `user` | The user message is used to provide the input to the model. |
| `assistant` | Output by the model which can either be a tool call or a message output. The output might also be associated with a particular “channel” identifying what the intent of the message is. |
| `analysis` | These are messages that are being used by the model for its chain-of-thought |
| `final` | Messages tagged in the final channel are messages intended to be shown to the end-user and represent the responses from the model. |
| `messages` | The list of messages that combine the content of the above to produce a full conversation. This is the input to the model. |

如果你熟悉 [OpenAI 的 messages 格式](https://platform.openai.com/docs/api-reference/messages/object)，會覺得很像，但有一個重要差異：

> `assistant` 這一輪有兩個特殊欄位：`thinking` 放模型的 reasoning 過程，`content` 放給使用者的最終回應。

要 fine-tune 模型，得先把這些 messages 轉成模型看得懂的格式。實務上就是用模型的 [_chat template_](https://huggingface.co/docs/transformers/chat_templating) 把每則訊息格式化，再把結果 tokenize。TRL 會自動做這件事，不過我們一步一步走一遍，搞懂它怎麼運作。

先載入 tokenizer：

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("openai/gpt-oss-20b")

然後用 tokenizer 的 `apply_chat_template()` 方法來格式化 messages：

In [ ]:
messages = dataset[0]["messages"]
conversation = tokenizer.apply_chat_template(messages, tokenize=False)
print(conversation)

這個 chat template 蠻精細的，仔細看一下。首先有特殊 token `<|start|>` 和 `<|end|>` 標示每則訊息的開頭和結尾，還有 `<|return|>` 標示整段對話結束。這些 token 幫模型理解對話結構。

可以看到有_兩種_ system 訊息：

* 預設的 `system`，所有訊息都會帶。上面的例子裡就是 _"You are ChatGPT, a large language model trained by OpenAI..."_ 那段。
* 特別的 `developer`，放自訂指令（對應我們 `messages` 物件裡的 `system` role）。這樣能針對單次對話補充模型該怎麼表現。上面的例子裡就是 _"You are an AI chatbot with a lively and energetic personality."_ 那段。

最後，assistant 的回應被放在一連串 _channel_ 裡：

* `analysis` channel 放模型的 reasoning 過程，讓它針對使用者的問題一步一步想。上面的例子裡就是法文那段 _"D'accord, l'utilisateur demande les tendances Twitter..."_。
* `final` channel 放給使用者看的最終回應。上面的例子裡就是 _"Hey there!  While I can't check Twitter..."_。

資料會怎麼處理搞清楚了，接下來準備模型。

## 準備模型

準備訓練的第一步是從 [Hugging Face Hub](https://huggingface.co) 下載權重。我們用 🤗 Transformers 的 `AutoModelForCausalLM` 來載入模型：

In [ ]:
import torch
from transformers import AutoModelForCausalLM, Mxfp4Config

quantization_config = Mxfp4Config(dequantize=True)
model_kwargs = dict(
    attn_implementation="eager",
    torch_dtype=torch.bfloat16,
    quantization_config=quantization_config,
    use_cache=False,
    device_map="auto",
)

model = AutoModelForCausalLM.from_pretrained("openai/gpt-oss-20b", **model_kwargs)

這樣會用訓練需要的設定載入模型。`attn_implementation` 設成 `eager` 效能比較好，`use_cache` 設成 `False` 是因為我們會搭配 gradient checkpointing 來 fine-tune。

如果你熟悉 🤗 Transformers，可能會注意到我們用了 `Mxfp4Config` 做量化。這是 OpenAI 模型專用的設定，讓我們能用混合精度訓練，搭配一種叫 [MXFP4](https://en.wikipedia.org/wiki/Block_floating_point) 的特殊 4-bit 浮點格式，這個格式針對 AI 工作負載做過優化。

訓練前，先用預設設定產生一個回應，看看模型原本的行為。做法是把範例 prompt tokenize，再讓模型產生回應：

In [ ]:
messages = [
    {"role": "user", "content": "¿Cuál es el capital de Australia?"},
]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)

output_ids = model.generate(input_ids, max_new_tokens=512)
response = tokenizer.batch_decode(output_ids)[0]
print(response)

這個例子裡，模型先用英文思考問題，最後用西班牙文回答。這是模型的預設行為，我們來看看能不能靠 fine-tuning 改掉它。

我們會用 [LoRA](https://huggingface.co/learn/llm-course/chapter11/4)（Low-Rank Adaptation）來 fine-tune。這個技巧只調整模型中少數特定的層，對 `openai/gpt-oss-20b` 這種大模型特別有用。

先把模型包成 `PeftModel`，並定義 LoRA 設定。這裡用 [PEFT library](https://github.com/huggingface/peft) 的 `LoraConfig` 類別：

In [ ]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules="all-linear",
    target_parameters=[
        "7.mlp.experts.gate_up_proj",
        "7.mlp.experts.down_proj",
        "15.mlp.experts.gate_up_proj",
        "15.mlp.experts.down_proj",
        "23.mlp.experts.gate_up_proj",
        "23.mlp.experts.down_proj",
    ],
)
peft_model = get_peft_model(model, peft_config)
peft_model.print_trainable_parameters()

這裡的 LoRA 超參數都是基本設定，你可以試試別的值，看對模型表現有什麼影響。例如把 `r` 調大，可訓練的參數會變多，模型可能更好，代價是需要更多 VRAM 和訓練時間。

**Note：** `openai/gpt-oss-20b` 是 [Mixture-of-Experts (MoE)](https://huggingface.co/blog/moe) 架構。除了指定 attention 層（`target_modules="all-linear"`），也要把 expert 模組裡的 projection 層納入。PEFT 用 `target_parameters` 參數支援這件事，可以指定像 `mlp.experts.down_proj` 和 `mlp.experts.gate_up_proj` 這類專家層。這個範例只選了其中一部分 projection 層，你可以自己試不同的組合。

模型和資料都準備好了，接下來定義訓練的超參數。

## Fine-tuning

TRL 提供 `SFTConfig` 類別來方便地定義訓練超參數。我們這樣設定 learning rate、batch size、epoch 數等參數：

In [ ]:
from trl import SFTConfig

training_args = SFTConfig(
    learning_rate=2e-4,
    gradient_checkpointing=True,
    num_train_epochs=1,
    logging_steps=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    max_length=2048,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine_with_min_lr",
    lr_scheduler_kwargs={"min_lr_rate": 0.1},
    output_dir="gpt-oss-20b-multilingual-reasoner",
    report_to="trackio",
    push_to_hub=True,
)

注意 `per_device_train_batch_size` 設成 4，`gradient_accumulation_steps` 設成 4，所以在 1 張 GPU 上等效的 batch size 是 4 x 4 = 16。你可能要依自己的硬體調整。我們也用 [Trackio](https://huggingface.co/blog/trackio) 記錄訓練進度和指標，你也可以換成別的 logging 工具。

訓練需要的東西都齊了。我們用 TRL 的 `SFTTrainer` 類別處理訓練流程，它會負責格式化資料集、套用 chat template 和訓練模型：

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)
trainer.train()

在 H100 上大約訓練 18 分鐘，其他硬體可能更久。

## 儲存模型並 push 到 Hugging Face Hub

最後可以把 fine-tune 好的模型 push 到你的 Hub repository 跟社群分享：

In [ ]:
trainer.save_model(training_args.output_dir)
trainer.push_to_hub(dataset_name="HuggingFaceH4/Multilingual-Thinking")

**Note**：為了避免 out-of-memory (OOM)，建議這時候重啟 kernel（Colab 就是重啟 runtime）。訓練完的模型還佔著 GPU 記憶體，但已經用不到了。

## 推論

模型上傳到 Hub 之後就能拿來做推論。先初始化原本的 base model 和 tokenizer，接著把 fine-tune 的權重合併進 base model，推論才會快：

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("openai/gpt-oss-20b")

# Load the original model first
model_kwargs = dict(attn_implementation="eager", torch_dtype="auto", use_cache=True, device_map="auto")
base_model = AutoModelForCausalLM.from_pretrained("openai/gpt-oss-20b", **model_kwargs).cuda()

# Merge fine-tuned weights with the base model
peft_model_id = "gpt-oss-20b-multilingual-reasoner"
model = PeftModel.from_pretrained(base_model, peft_model_id)
model = model.merge_and_unload()

模型載入完成，最後一步是讓它產生 token。這裡用模型的 `generate` 方法，根據輸入 prompt 產生輸出。先定義 prompt：

接著 tokenize prompt、產生輸出，最後把輸出 token decode 成最終回應：

In [ ]:
REASONING_LANGUAGE = "German"
SYSTEM_PROMPT = f"reasoning language: {REASONING_LANGUAGE}"
USER_PROMPT = "¿Cuál es el capital de Australia?"  # Spanish for "What is the capital of Australia?"

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT},
]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)

gen_kwargs = {"max_new_tokens": 512, "do_sample": True, "temperature": 0.6, "top_p": None, "top_k": None}

output_ids = model.generate(input_ids, **gen_kwargs)
response = tokenizer.batch_decode(output_ids)[0]
print(response)

再試試模型沒有明確 fine-tune 過的語言，例如中文和印地語：

In [ ]:
REASONING_LANGUAGE = "Chinese"  # or Hindi, or any other language...
SYSTEM_PROMPT = f"reasoning language: {REASONING_LANGUAGE}"
USER_PROMPT = "What is the national symbol of Canada?"

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT},
]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)

output_ids = model.generate(input_ids, **gen_kwargs)
response = tokenizer.batch_decode(output_ids)[0]
print(response)

成功了，我們已經把 `openai/gpt-oss-20b` fine-tune 成能用多種語言做 reasoning。

## 結論

恭喜，你已經用 TRL library 和 LoRA 成功 fine-tune 出一個多語言 reasoning 模型。這份 notebook 的步驟可以套用到 Hugging Face Hub 上很多其他 [datasets](https://huggingface.co/datasets)，拿來 fine-tune [`openai/gpt-oss-20b`](https://huggingface.co/openai/gpt-oss-20b)，期待看到你做出什麼。

## 重點整理
- Harmony 格式把對話拆成 `developer`、`user`、`assistant`，assistant 的輸出再分 `analysis`（reasoning）和 `final`（給使用者的回答）兩個 channel。
- Fine-tuning 前先看 chat template 產出的文字，就知道模型實際看到什麼。
- LoRA 只訓練少量參數，記憶體需求低很多。gpt-oss 是 MoE，要用 `target_parameters` 把 expert 層也納入。
- `SFTTrainer` 會自動套 chat template 和 tokenize，你主要調的是 `LoraConfig` 和 `SFTConfig`。
- 推論前要把 LoRA 權重合併進 base model，而且訓練完最好重啟 runtime 釋放 GPU 記憶體。

## 練習
1. 把 `SFTConfig` 的 `per_device_train_batch_size` 和 `gradient_accumulation_steps` 對調（例如 4/4 改成 2/8），確認等效 batch size 不變，看訓練速度和記憶體用量有什麼差別。
2. 把 `LoraConfig` 的 `r` 改成原本的兩倍，觀察可訓練參數量、VRAM 和 loss 怎麼變。
3. 改 `target_parameters`，多加或拿掉一個 expert projection 層，比較 fine-tune 後在中文、印地語 prompt 上的 reasoning 語言有沒有差異。